# Imports

In [1]:
# SwarmRL Imports
import swarmrl as srl
from swarmrl.models.interaction_model import Action
from swarmrl.observables import Observable
import swarmrl.engine.espresso as espresso
from swarmrl.utils import utils

# PySCF imports
import pyscf
from pyscf import gto, scf, dft
from pyscf.geomopt.geometric_solver import optimize

# Linalg imports
import flax.linen as nn
import numpy as np
import optax
import jax.numpy as jnp
import jax

# Productivity imports
import h5py as hf
import matplotlib.pyplot as plt
import pint
from typing import List

## Simulation

In [2]:
simulation_name = "min-finding"
seed = np.random.randint(0, 3453276453)
temperature = 150.0
n_colloids = 5

ureg = pint.UnitRegistry()
md_params = espresso.MDParams(
    ureg=ureg,
    fluid_dyn_viscosity=ureg.Quantity(8.9e-4, "pascal * second"),
    WCA_epsilon=ureg.Quantity(273.15, "kelvin") * ureg.boltzmann_constant,
    temperature=ureg.Quantity(temperature, "kelvin"),
    box_length=ureg.Quantity(360, "micrometer"),
    time_slice=ureg.Quantity(1.0, "second"),  # model timestep
    time_step=ureg.Quantity(0.01, "second"),  # integrator timestep
    write_interval=ureg.Quantity(2, "second"),
)

system_runner = srl.espresso.EspressoMD(
    md_params=md_params,
    n_dims=2,
    seed=seed,
    out_folder='./alanine-dipeptide',
    write_chunk_size=100,
)

system_runner.add_colloids(
    n_colloids,
    ureg.Quantity(2.14, "micrometer"),
    ureg.Quantity(np.array([500, 500, 0]), "micrometer"),
    ureg.Quantity(250, "micrometer"),
    type_colloid=0,
)

In [3]:
n_episodes = 300
episode_length = 5

## Reinforcement Learning

In [4]:
# Exploration policy
exploration_policy = srl.exploration_policies.RandomExploration(probability=0.2)

# Sampling strategy
sampling_strategy = srl.sampling_strategies.GumbelDistribution()

# Value function
value_function = srl.value_functions.ExpectedReturns(
    gamma=0.99, standardize=True
)


In [5]:
class MinimaDetection(srl.tasks.Task):
    """
    Task for minima detection.
    """
    def __init__(
        self,
        potential_fn: callable,
        particle_type: int = 0,
    ):
        """
        Constructor for the find origin task.

        Parameters
        ----------
        source : np.ndarray (default = (0, 0 0))
                Source of the gradient.
        potenial_fn : callable
                Function we are moving along.
        particle_type : int (default=0)

        """
        super().__init__(particle_type=particle_type)
        self.potential_fn = potential_fn

        # Class only attributes
        self._historic_potential = {}
        
    def initialize(self, colloids: List):
        """
        Prepare the task for running.

        Parameters
        ----------
        colloids : List[Colloid]
                List of colloids to be used in the task.

        Returns
        -------
        observable :
                Returns the observable required for the task.
        """
        for item in colloids:
            if item.type == self.particle_type:
                index = item.id
                function_value = self.potential_fn(item.pos)
                self._historic_potential[str(index)] = function_value
                
    def compute_colloid_reward(self, index: int, colloids):
        """
        Compute the reward for a single colloid.

        Parameters
        ----------
        index : int
                Index of the colloid to compute the reward for.
        colloids : list
                All colloids

        Returns
        -------
        reward : float
                Reward for the colloid.
        """
        colloid_id = colloids[index].id
        # Get the current position of the colloid
        current_position =colloids[index].pos
        current_function_value = self.potential_fn(current_position)
        past_function_value = self._historic_potential[str(colloid_id)]

        # Compute difference in scaled_distances
        delta = current_function_value - past_function_value

        # Compute the reward
        reward = -100 * delta + abs(current_function_value)

        # Update the historic position
        self._historic_potential[str(colloid_id)] = current_function_value

        return reward
    
    def __call__(self, colloids: List):
        """
        Compute the reward.

        In this case of this task, the observable itself is the gradient of the field
        that the colloid is swimming in. Therefore, the change is simply scaled and
        returned.

        Parameters
        ----------
        colloids : List[Colloid] (n_colloids, )
                List of colloids to be used in the task.

        Returns
        -------
        rewards : List[float] (n_colloids, )
                Rewards for each colloid.
        """
        colloid_indices = self.get_colloid_indices(colloids)

        return np.array(
            [self.compute_colloid_reward(index, colloids) for index in colloid_indices]
        )


In [6]:
class MDObservable(srl.observables.Observable):
    def __init__(
        self,
        potential_fn: callable,
        particle_type: int = 0,
    ):
        """
        Constructor for the find origin task.

        Parameters
        ----------
        source : np.ndarray (default = (0, 0 0))
                Source of the gradient.
        potenial_fn : callable
                Function we are moving along.
        particle_type : int (default=0)

        """
        super().__init__(particle_type=particle_type)
        self.potential_fn = potential_fn

        # Class only attributes
        self._historic_potential = {}
        
    def initialize(self, colloids: List):
        """
        Prepare the task for running.

        Parameters
        ----------
        colloids : List[Colloid]
                List of colloids to be used in the task.

        Returns
        -------
        observable :
                Returns the observable required for the task.
        """
        for item in colloids:
            if item.type == self.particle_type:
                index = item.id
                function_value = self.potential_fn(item.pos)
                self._historic_potential[str(index)] = function_value
                
    def compute_single_observable(self, index: int, colloids: List) -> float:
        """
        Compute the observable for a single colloid.

        Parameters
        ----------
        index : int
                Index of the colloid to compute the observable for.
        colloids : List[Colloid]
                List of colloids in the system.
        """
        colloid_id = colloids[index].id
        # Get the current position of the colloid
        current_position =colloids[index].pos
        current_function_value = self.potential_fn(current_position)
        past_function_value = self._historic_potential[str(colloid_id)]

        # Compute difference in scaled_distances
        delta = current_function_value - past_function_value

        # Update the historic position
        self._historic_potential[str(colloid_id)] = current_function_value

        return 10 * delta
    
    def compute_observable(self, colloids: List):
        """
        Compute the position of the colloid.

        Parameters
        ----------
        colloids : List[Colloid] (n_colloids, )
                List of all colloids in the system.

        Returns
        -------
        observables : List[float] (n_colloids, dimension)
                List of observables, one for each colloid. In this case,
                current field value minus to previous field value.
        """
        reference_ids = self.get_colloid_indices(colloids)

        observables = [
            self.compute_single_observable(index, colloids) for index in reference_ids
        ]

        return np.array(observables).reshape(-1, 1)


In [7]:
def compute_energy(coordinates: np.ndarray) -> float:
    """
    Compute the energy of the protein.
    
    Steps
    -----
    1. Constrain angles
    2. Perform geometry minimization
    3. Compute ground state energy
    
    Parameters
    ----------
    phi (coordinates[0]) : float
            Phi angle
    psi (coordinates[1]) : float
            Psi angle
            
    Returns
    -------
    energy : float
            Ground state energy of the protein.
    """
    # standard AD geometry.
    pyscf_input = '''
         H  3.225  27.427  2.566; 
         C  3.720  26.570  2.110; 
         H  4.088  25.905  2.891; 
         H  4.557  26.914  1.502; 
         C  2.770  25.800  1.230; 
         O  1.600  26.150  1.090;
         N  3.270  24.640  0.690; 
         H  4.259  24.471  0.810; 
         C  2.480  23.690  -0.190; 
         H  1.733  24.315  -0.679; 
         C  3.470  23.160  -1.270; 
         H  4.219  22.525  -0.797; 
         H  2.922  22.582  -2.014; 
         H  3.963  24.002  -1.756; 
         C  1.730  22.590  0.490; 
         O  2.340  21.880  1.280;
         N  0.400  22.430  0.210;
         H  -0.008  23.118  -0.407; 
         C  -0.470  21.350  0.730; 
         H  0.112  20.693  1.376; 
         H  -1.290  21.786  1.300; 
         H  -0.873  20.775  -0.103;
     '''
    
    # Define the constraint file.
    params = {"constraints": "constraints.txt",}
    
    # Create the constraint file.
    lines = [
        ["$set"], 
        [f"dihedral 5 7 9 15 {coordinates[0] - 180.}"], 
        ["$set"], 
        [f"dihedral 7 9 15 17 {coordinates[1] - 180.}"]
    ]

    with open("constraints.txt", "w") as f:
        for line in lines:
            f.write(f"{line[0]}\n")
    
    # Perform geometry minimzation.
    molecule = gto.M(atom=pyscf_input, basis='ccpvdz')
    calculator = dft.RKS(molecule)
    equilibrated_molecule = optimize(calculator, maxsteps=1, **params)
    
    # Compute ground state energy.
    equilibrate_calculator = dft.RKS(equilibrated_molecule)
    _ = equilibrate_calculator.kernel()
    gs_energy = equilibrate_calculator.e_tot
    
    return gs_energy

observable = MDObservable(compute_energy)
task = MinimaDetection(compute_energy)
    

In [8]:
loss = srl.losses.PolicyGradientLoss(value_function=value_function)

In [9]:
class ActorNet(nn.Module):
    """A simple dense model."""

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)
        return x

class CriticNet(nn.Module):
    """A simple dense model."""

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=1)(x)
        return x

In [10]:
actor = srl.networks.FlaxModel(
    flax_model=ActorNet(),
    optimizer=optax.adam(learning_rate=0.001),
    input_shape=(1,),
    sampling_strategy=sampling_strategy,
    exploration_policy=exploration_policy,
)

critic = srl.networks.FlaxModel(
    flax_model=CriticNet(),
    optimizer=optax.adam(learning_rate=0.001),
    input_shape=(1,),
)

In [11]:
translate = Action(force=10.0)
rotate_clockwise = Action(torque=np.array([0.0, 0.0, 10.0]))
rotate_counter_clockwise = Action(torque=np.array([0.0, 0.0, -10.0]))
do_nothing = Action()

actions = {
    "RotateClockwise": rotate_clockwise,
    "Translate": translate,
    "RotateCounterClockwise": rotate_counter_clockwise,
    "DoNothing": do_nothing,
}

In [12]:
protocol = srl.rl_protocols.ActorCritic(
    particle_type=0, 
    actor=actor, 
    critic=critic, 
    task=task, 
    observable=observable, 
    actions=actions
)

In [13]:
rl_trainer = srl.gyms.Gym(
    [protocol],
    loss,
)
rl_trainer.restore_models()

## Model Running

In [14]:
rl_trainer.perform_rl_training(
    system_runner=system_runner,
    n_episodes=n_episodes,
    episode_length=episode_length,
)

geometric-optimize called with the following command line:
/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/ipykernel_launcher.py -f /Users/samueltovey/Library/Jupyter/runtime/kernel-6bab89b1-ca09-4391-8b4a-b99520c81d37.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   H   3.225000  27.427000   2.566000    0.000000  0.000000  0.000000
   C   3.720000  26.570000   2.110000    0.000000  0.000000  0.000000
   H   4.088000  25.905000   2.891000    0.000000  0.000000  0.000000
   H   4.557000  26.914000   1.502000    0.000000  0.000000  0.000000
   C   2.770000  25.800000   1.230000    0.000000  0.000000  0.000000
   O   1.600000  26.150000   1.090000    0.000000  0.000000  0.000000
   N   3.270000  24.640000   0.690000    0.000000  0.000000  0.000000
   H   4.259000  24.471000   0.810000    0.000000  0.000000  0.000000
   C   2.480000  23.690000  -0.190000   -0.000000  0.000000  0.000000
   H   1.733000  24.315000  -0.679000    0.000000  0.000000  0.000000
   C   3.470000  23.160000  -1.270000    0.000000  0.000000  0.000000
   H   4.219000  22.525000  -0.797000    0.000000  0.000000  0.000000
   H   2.922000  22.582000  -2.0

Step    0 : Gradient = 2.262e-02/4.627e-02 (rms/max) Energy = -491.7165958322
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 5.12583e-01 9.14078e-01 9.32588e-01


Geometry optimization is not converged in 1 iterations
converged SCF energy = -491.716595832164


geometric-optimize called with the following command line:
/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/ipykernel_launcher.py -f /Users/samueltovey/Library/Jupyter/runtime/kernel-6bab89b1-ca09-4391-8b4a-b99520c81d37.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      

...

Step    0 : Gradient = 2.262e-02/4.627e-02 (rms/max) Energy = -491.7165958322
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 5.12583e-01 9.14078e-01 9.32588e-01


Geometry optimization is not converged in 1 iterations
converged SCF energy = -491.716595832162


geometric-optimize called with the following command line:
/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/ipykernel_launcher.py -f /Users/samueltovey/Library/Jupyter/runtime/kernel-6bab89b1-ca09-4391-8b4a-b99520c81d37.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   H   3.225000  27.427000   2.566000    0.000000  0.000000  0.000000
   C   3.720000  26.570000   2.110000    0.000000  0.000000  0.000000
   H   4.088000  25.905000   2.891000    0.000000  0.000000  0.000000
   H   4.557000  26.914000   1.502000    0.000000  0.000000  0.000000
   C   2.770000  25.800000   1.230000    0.000000  0.000000  0.000000
   O   1.600000  26.150000   1.090000    0.000000  0.000000  0.000000
   N   3.270000  24.640000   0.690000    0.000000  0.000000  0.000000
   H   4.259000  24.471000   0.810000    0.000000  0.000000  0.000000
   C   2.480000  23.690000  -0.190000   -0.000000  0.000000  0.000000
   H   1.733000  24.315000  -0.679000    0.000000  0.000000  0.000000
   C   3.470000  23.160000  -1.270000    0.000000  0.000000  0.000000
   H   4.219000  22.525000  -0.797000    0.000000  0.000000  0.000000
   H   2.922000  22.582000  -2.0


KeyboardInterrupt



In [ ]:
rl_trainer.export_models()